# Второй тестовый парсер: продажи по контрагентам

Файл-источник: `Данные/Продажи по контрагентам/Продажи по контрагентам за Январь 2026 г. - Апрель 2026 г..xls`.

Итоговая структура: одна строка = одна позиция номенклатуры в документе продажи/корректировки на дату.


In [ ]:

from pathlib import Path
import importlib.util
import re
import sys

import pandas as pd

# Для старых .xls нужен xlrd.
# Если xlrd не установлен, выполни:
# %pip install xlrd openpyxl
if importlib.util.find_spec("xlrd") is None:
    temp_xlrd = Path("/private/tmp/codex-xlrd")
    if temp_xlrd.exists():
        sys.path.insert(0, str(temp_xlrd))

if importlib.util.find_spec("xlrd") is None:
    raise ModuleNotFoundError("Нужен пакет xlrd: выполни `%pip install xlrd openpyxl`")


SOURCE_FILE = Path("Данные/Продажи по контрагентам/Продажи по контрагентам за Январь 2026 г. - Апрель 2026 г..xls")
OUTPUT_XLSX = Path("Данные/sales_by_counterparties_second_try.xlsx")


def clean_text(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    return re.sub(r"\s+", " ", text) or None


def parse_date_ddmmyy(value):
    text = clean_text(value)
    if not text:
        return None
    return pd.to_datetime(text, format="%d.%m.%y", errors="coerce")


def parse_date_ddmmyyyy(value):
    if not value:
        return pd.NaT
    return pd.to_datetime(value, format="%d.%m.%Y", errors="coerce")


def is_contract(text):
    text = clean_text(text)
    return bool(text and re.search(r"\s+от\s+\d{2}\.\d{2}\.\d{4}$", text))


def is_document(text):
    text = clean_text(text)
    if not text:
        return False
    return text.startswith("Реализация ") or text.startswith("Корректировка реализации ")


def is_counterparty(text):
    text = clean_text(text)
    if not text:
        return False
    lowered = text.lower()
    return "магазин светофор" in lowered or "магазин маяк" in lowered


def is_stop_row(text):
    text = clean_text(text)
    return (not text) or text == "Итого" or text.startswith("*")


def parse_contract(contract_raw):
    contract_raw = clean_text(contract_raw)
    if not contract_raw:
        return {"contract_number": None, "contract_date": pd.NaT}
    match = re.search(r"\s+от\s+(\d{2}\.\d{2}\.\d{4})$", contract_raw)
    if not match:
        return {"contract_number": contract_raw, "contract_date": pd.NaT}
    return {
        "contract_number": contract_raw[:match.start()].strip(),
        "contract_date": parse_date_ddmmyyyy(match.group(1)),
    }


def parse_document(document_raw):
    document_raw = clean_text(document_raw)
    result = {
        "document_type": None,
        "document_number": None,
        "document_date": pd.NaT,
    }
    if not document_raw:
        return result

    if document_raw.startswith("Реализация "):
        result["document_type"] = "Реализация"
    elif document_raw.startswith("Корректировка реализации "):
        result["document_type"] = "Корректировка реализации"

    match = re.search(r"№\s*([^\s]+)\s+от\s+(\d{2}\.\d{2}\.\d{4})", document_raw)
    if match:
        result["document_number"] = match.group(1)
        result["document_date"] = parse_date_ddmmyyyy(match.group(2))
    return result


def parse_counterparty(counterparty_raw):
    counterparty_raw = clean_text(counterparty_raw)
    result = {
        "legal_entity": None,
        "brand": None,
        "store_location_raw": None,
        "city_or_area": None,
    }
    if not counterparty_raw:
        return result

    lowered = counterparty_raw.lower()
    if "магазин светофор" in lowered:
        match = re.search(r"магазин\s+светофор", counterparty_raw, flags=re.IGNORECASE)
        left = counterparty_raw[:match.start()]
        location = counterparty_raw[match.end():]
        result["legal_entity"] = left.strip()
        result["brand"] = "Светофор"
        result["store_location_raw"] = location.strip(" ,") or None
    elif "магазин маяк" in lowered:
        match = re.search(r"магазин\s+маяк", counterparty_raw, flags=re.IGNORECASE)
        left = counterparty_raw[:match.start()]
        location = counterparty_raw[match.end():]
        result["legal_entity"] = left.strip().strip(",")
        result["brand"] = "Маяк"
        result["store_location_raw"] = location.strip(" ,") or None
    else:
        result["legal_entity"] = counterparty_raw

    location = result["store_location_raw"] or ""
    city_match = re.search(r"((?:г\.|рп\.|пгт|с\.)\s*[^,]+)", location)
    if city_match:
        result["city_or_area"] = city_match.group(1).strip()
    return result


def build_date_pairs(raw):
    pairs = []
    headers = raw.iloc[0].tolist()
    subheaders = raw.iloc[1].tolist()

    # Колонки 1 и 2 - общий итог: Количество и Сумма.
    for qty_col in range(3, len(headers), 2):
        amount_col = qty_col + 1
        if amount_col >= len(headers):
            continue
        date = parse_date_ddmmyy(headers[qty_col])
        if pd.isna(date):
            continue
        if clean_text(subheaders[qty_col]) != "Количество" or clean_text(subheaders[amount_col]) != "Сумма":
            continue
        pairs.append((date, qty_col, amount_col))
    return pairs


def parse_sales_report(path):
    path = Path(path)
    raw = pd.read_excel(path, sheet_name=0, header=None, engine="xlrd")
    date_pairs = build_date_pairs(raw)

    rows = []
    i = 4
    while i < len(raw):
        counterparty_raw = clean_text(raw.iat[i, 0])
        if is_stop_row(counterparty_raw):
            break

        counterparty = parse_counterparty(counterparty_raw)
        j = i + 1
        while j < len(raw):
            contract_raw = clean_text(raw.iat[j, 0])
            if is_stop_row(contract_raw) or not is_contract(contract_raw):
                break

            contract = parse_contract(contract_raw)
            j += 1

            while j < len(raw):
                document_raw = clean_text(raw.iat[j, 0])
                if not is_document(document_raw):
                    break

                document = parse_document(document_raw)
                j += 1

                # После документа идут строки номенклатуры до следующего документа/договора/контрагента.
                while j < len(raw):
                    item_name = clean_text(raw.iat[j, 0])
                    if (
                        is_stop_row(item_name)
                        or is_document(item_name)
                        or is_contract(item_name)
                        or is_counterparty(item_name)
                    ):
                        break

                    for column_date, qty_col, amount_col in date_pairs:
                        quantity = raw.iat[j, qty_col]
                        amount = raw.iat[j, amount_col]
                        if pd.isna(quantity) and pd.isna(amount):
                            continue
                        quantity = 0.0 if pd.isna(quantity) else float(quantity)
                        amount = 0.0 if pd.isna(amount) else float(amount)
                        if quantity == 0 and amount == 0:
                            continue

                        sale_date = column_date
                        if pd.notna(document["document_date"]) and document["document_date"] != column_date:
                            sale_date = document["document_date"]

                        rows.append({
                            "counterparty_raw": counterparty_raw,
                            **counterparty,
                            "contract_raw": contract_raw,
                            **contract,
                            "sales_doc_raw": document_raw,
                            "sales_doc_type": document["document_type"],
                            "sales_doc_number": document["document_number"],
                            "sales_doc_date": document["document_date"],
                            "sale_date": sale_date,
                            "nomenclature": item_name,
                            "quantity": quantity,
                            "amount": amount,
                            "source_file": path.name,
                            "source_sheet": "Лист_1",
                            "source_row_number": j + 1,
                            "source_quantity_column": raw.iat[0, qty_col],
                            "source_amount_column": raw.iat[0, amount_col - 1],
                        })

                    j += 1

        i = j

    columns = [
        "counterparty_raw",
        "legal_entity",
        "brand",
        "store_location_raw",
        "city_or_area",
        "contract_raw",
        "contract_number",
        "contract_date",
        "sales_doc_raw",
        "sales_doc_type",
        "sales_doc_number",
        "sales_doc_date",
        "sale_date",
        "nomenclature",
        "quantity",
        "amount",
        "source_file",
        "source_sheet",
        "source_row_number",
        "source_quantity_column",
        "source_amount_column",
    ]
    return pd.DataFrame(rows)[columns]


sales_df = parse_sales_report(SOURCE_FILE)
sales_df.shape


In [ ]:

sales_df.head(20)


## Проверка итогов

Сверяем сумму и количество с итоговой строкой Excel.

In [ ]:

print(f"Строк в sales_df: {len(sales_df):,}")
print(f"Количество: {sales_df['quantity'].sum():,.2f}")
print(f"Сумма:      {sales_df['amount'].sum():,.2f}")


In [ ]:

summary_by_month = (
    sales_df
    .assign(month=sales_df["sale_date"].dt.to_period("M").astype(str))
    .groupby("month", as_index=False)
    .agg(quantity=("quantity", "sum"), amount=("amount", "sum"))
    .sort_values("month")
)
summary_by_month


In [ ]:

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    sales_df.to_excel(writer, sheet_name="sales_items", index=False)
    summary_by_month.to_excel(writer, sheet_name="summary_by_month", index=False)
    (
        sales_df
        .groupby(["legal_entity", "brand"], dropna=False, as_index=False)
        .agg(quantity=("quantity", "sum"), amount=("amount", "sum"), documents=("sales_doc_raw", "nunique"))
        .sort_values("amount", ascending=False)
        .to_excel(writer, sheet_name="summary_by_group", index=False)
    )

OUTPUT_XLSX
